In [9]:
import argparse, os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image
import timm
import torchvision.transforms as T


In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXTS = (".jpg", ".jpeg", ".png", ".webp")

In [11]:
# 데이터
def df_from_folder(root):
    rows = []
    for d in sorted(os.listdir(root)):
        dd = os.path.join(root, d)
        if not os.path.isdir(dd):
            continue
        for f in sorted(os.listdir(dd)):
            if f.lower().endswith(EXTS):
                rows.append({"identity": d, "path": os.path.join(d, f)})
    return pd.DataFrame(rows)


def split_by_identity(df, val_frac=0.2, seed=0):
    rng = np.random.default_rng(seed)
    ids = df["identity"].unique().copy()
    rng.shuffle(ids)
    val_ids = set(ids[: int(len(ids) * val_frac)])
    tr = df[~df["identity"].isin(val_ids)].reset_index(drop=True)
    va = df[df["identity"].isin(val_ids)].reset_index(drop=True)
    return tr, va


class ReIDDataset(Dataset):
    def __init__(self, df, root, tf):
        self.paths = df["path"].tolist()
        self.y = df["label"].to_numpy() if "label" in df else np.zeros(len(df), int)
        self.root, self.tf = root, tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, self.paths[i])).convert("RGB")
        return self.tf(img), int(self.y[i])


class PKSampler(Sampler):
    """배치 = P개체 x K장. metric learning 필수."""
    def __init__(self, labels, p, k, steps=None):
        self.labels = np.asarray(labels)
        self.p, self.k = p, k
        self.by_id = {c: np.where(self.labels == c)[0] for c in np.unique(self.labels)}
        self.ids = list(self.by_id)
        self.steps = steps or max(1, len(self.labels) // (p * k))

    def __iter__(self):
        for _ in range(self.steps):
            picks = np.random.choice(self.ids, self.p, replace=len(self.ids) < self.p)
            batch = []
            for c in picks:
                idx = self.by_id[c]
                batch += list(np.random.choice(idx, self.k, replace=len(idx) < self.k))
            yield batch

    def __len__(self):
        return self.steps


In [12]:
# ArcFace
class ArcFaceHead(nn.Module):
    def __init__(self, in_dim, n_classes, s=64.0, m=0.5):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_classes, in_dim))
        nn.init.xavier_uniform_(self.W)
        self.s, self.m = s, m
        self.cos_m, self.sin_m = np.cos(m), np.sin(m)
        self.th, self.mm = np.cos(np.pi - m), np.sin(np.pi - m) * m

    def forward(self, feats, labels):
        cos = F.linear(F.normalize(feats), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt(1.0 - cos ** 2)
        phi = cos * self.cos_m - sin * self.sin_m           # cos(theta + m)
        phi = torch.where(cos > self.th, phi, cos - self.mm)  # 단조성 가드
        oh = F.one_hot(labels, cos.size(1)).float()
        return (oh * phi + (1.0 - oh) * cos) * self.s

In [13]:
# 평가
def _ap_cmc(rel):
    if not rel.any():
        return 0.0, 0, 0
    hits = np.cumsum(rel)
    ranks = np.arange(1, len(rel) + 1)
    return float((hits / ranks * rel).sum() / rel.sum()), int(rel[0]), int(rel[:5].any())


@torch.no_grad()
def extract(model, loader):
    model.eval()
    embs, labs = [], []
    for x, y in loader:
        v = F.normalize(model(x.to(DEVICE)))
        embs.append(v.cpu().numpy())
        labs.append(y.numpy())
    return np.concatenate(embs), np.concatenate(labs)


def evaluate_map_split(emb, labels, gallery_frac=0.5, seed=0):
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    g_idx, q_idx = [], []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0].copy()
        rng.shuffle(idx)
        if len(idx) <= 1:
            g_idx += idx.tolist(); continue
        k = max(1, round(len(idx) * gallery_frac))
        g_idx += idx[:k].tolist(); q_idx += idx[k:].tolist()
    g_idx, q_idx = np.array(g_idx), np.array(q_idx)
    g_emb, g_lab = emb[g_idx], labels[g_idx]
    sim = emb[q_idx] @ g_emb.T
    aps = c1 = c5 = 0.0
    for row, gt in zip(sim, labels[q_idx]):
        ap, h1, h5 = _ap_cmc(g_lab[np.argsort(row)[::-1]] == gt)
        aps += ap; c1 += h1; c5 += h5
    q = len(q_idx)
    return aps / q, c1 / q, c5 / q

In [14]:
# 학습
def set_trainable(model, freeze_until):
    """Swin timm: freeze_until = 학습시킬 첫 stage index. -1 = 전체 학습, 99 = 전부 freeze(헤드만)."""
    for p in model.parameters():
        p.requires_grad = True
    if freeze_until < 0:
        return
    stages = list(model.layers) if hasattr(model, "layers") else []
    for p in model.patch_embed.parameters():
        p.requires_grad = False
    for i, st in enumerate(stages):
        req = i >= freeze_until
        for p in st.parameters():
            p.requires_grad = req


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", help="개체당 폴더 루트")
    ap.add_argument("--csv", help="identity,path CSV (--data 대신)")
    ap.add_argument("--root", default=".", help="csv 사용 시 path 기준 루트")
    ap.add_argument("--model", default="hf-hub:BVRA/MegaDescriptor-B-224")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--pk-p", type=int, default=12)
    ap.add_argument("--pk-k", type=int, default=4)
    ap.add_argument("--lr-head", type=float, default=1e-4)
    ap.add_argument("--lr-backbone", type=float, default=1e-5)
    ap.add_argument("--wd", type=float, default=1e-4)
    ap.add_argument("--freeze-until", type=int, default=2,
                    help="학습시킬 첫 Swin stage. -1 전체, 2 뒤쪽만, 99 헤드만")
    ap.add_argument("--val-frac", type=float, default=0.2)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="ML/megadescriptor_ft.pt")
    args = ap.parse_args()

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)

    # 데이터
    if args.csv:
        df = pd.read_csv(args.csv); root = args.root
    else:
        df = df_from_folder(args.data); root = args.data
    tr, va = split_by_identity(df, args.val_frac, args.seed)
    id2lab = {c: i for i, c in enumerate(sorted(tr["identity"].unique()))}
    tr["label"] = tr["identity"].map(id2lab)
    va = va.copy()
    va["label"] = va["identity"].astype("category").cat.codes  # 평가용 임시 라벨
    n_classes = len(id2lab)
    print(f"train {len(tr)}장 / {n_classes}개체  |  val {len(va)}장 / {va['identity'].nunique()}개체")

    # 모델 + transform
    model = timm.create_model(args.model, pretrained=True, num_classes=0).to(DEVICE)
    cfg = timm.data.resolve_model_data_config(model)
    size = cfg["input_size"][-1]
    train_tf = T.Compose([
        T.RandomResizedCrop(size, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.2, 0.2, 0.2, 0.05),
        T.ToTensor(),
        T.Normalize(cfg["mean"], cfg["std"]),
    ])
    eval_tf = timm.data.create_transform(**cfg, is_training=False)

    set_trainable(model, args.freeze_until)
    head = ArcFaceHead(model.num_features, n_classes).to(DEVICE)

    # 로더
    tr_ds = ReIDDataset(tr, root, train_tf)
    tr_ld = DataLoader(tr_ds, batch_sampler=PKSampler(tr["label"].to_numpy(), args.pk_p, args.pk_k),
                       num_workers=4, pin_memory=True)
    va_ld = DataLoader(ReIDDataset(va, root, eval_tf), batch_size=64, num_workers=4, pin_memory=True)

    # 옵티마이저 (backbone / head 차등 LR)
    bb = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(
        [{"params": bb, "lr": args.lr_backbone},
         {"params": head.parameters(), "lr": args.lr_head}],
        weight_decay=args.wd,
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    scaler = torch.amp.GradScaler(DEVICE)

    best = -1.0
    for ep in range(1, args.epochs + 1):
        model.train(); head.train()
        losses = []
        for x, y in tr_ld:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast(DEVICE):
                logits = head(model(x), y)
                loss = F.cross_entropy(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            losses.append(loss.item())
        sched.step()

        emb, lab = extract(model, va_ld)
        mAP, t1, t5 = evaluate_map_split(emb, lab, seed=args.seed)
        print(f"[{ep:02d}/{args.epochs}] loss {np.mean(losses):.3f} | "
              f"val mAP {mAP:.4f}  Top-1 {t1:.4f}  Top-5 {t5:.4f}")

        if mAP > best:
            best = mAP
            torch.save({"model": args.model, "backbone": model.state_dict(),
                        "id2lab": id2lab, "cfg": cfg, "val_mAP": best}, args.out)
            print(f"    -> saved {args.out} (mAP {best:.4f})")

    print(f"\n완료. best val mAP {best:.4f}")

In [16]:
main()

usage: ipykernel_launcher.py [-h] [--data DATA] [--csv CSV] [--root ROOT]
                             [--model MODEL] [--epochs EPOCHS] [--pk-p PK_P]
                             [--pk-k PK_K] [--lr-head LR_HEAD]
                             [--lr-backbone LR_BACKBONE] [--wd WD]
                             [--freeze-until FREEZE_UNTIL]
                             [--val-frac VAL_FRAC] [--seed SEED] [--out OUT]
ipykernel_launcher.py: error: argument --freeze-until: invalid int value: 'c:\\Users\\Admin\\AppData\\Roaming\\jupyter\\runtime\\kernel-v368f26ee50b9426e8a332e23ce8d634c69f4eb28b.json'


SystemExit: 2